# API Integration — PySpark + Delta Lake version (Databricks)

Spark/Delta-native counterpart to `api_integration_reference.ipynb`.

Pagination, retry-with-backoff and rate limiting stay **driver-side Python** —
a cursor-paginated REST API is inherently sequential (each page's cursor
depends on the previous response), so there's nothing to distribute across
executors. What changes for Databricks is what happens *around* that loop:

- fetched pages land in a Delta **bronze** table instead of a Python list
- the pagination cursor is checkpointed to a Delta **watermark** table, so a
  rerun resumes instead of re-fetching everything
- `IdempotentProcessor` (an in-memory dict) is replaced by a Delta
  **`MERGE INTO`** upsert — the standard idempotent-write pattern on Databricks

Run on a Databricks cluster, or locally with `pyspark` + `delta-spark`.

## Setup

In [1]:
import time, random, functools, threading
from pyspark.sql import SparkSession, functions as F
from delta.tables import DeltaTable

try:
    spark
except NameError:
    from delta import configure_spark_with_delta_pip
    builder = (SparkSession.builder
               .appName("api_integration_reference")
               .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
               .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog"))
    spark = configure_spark_with_delta_pip(builder).getOrCreate()

try:
    dbutils
except NameError:
    dbutils = None

In [2]:
if dbutils is not None:
    dbutils.widgets.text("catalog", "", "Unity Catalog (blank = hive_metastore)")
    dbutils.widgets.text("schema", "devrev_ref", "Schema name")
    CATALOG = dbutils.widgets.get("catalog").strip()
    SCHEMA = dbutils.widgets.get("schema").strip() or "devrev_ref"
else:
    CATALOG = ""
    SCHEMA = "devrev_ref"

SCHEMA_FQN = f"{CATALOG}.{SCHEMA}" if CATALOG else SCHEMA

def qualified(table):
    return f"{SCHEMA_FQN}.{table}"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA_FQN}")
print("Delta tables will be written under:", SCHEMA_FQN)

Box(children=(Label(value='Unity Catalog (blank = hive_metastore)'), Text(value='')))

Box(children=(Label(value='Schema name'), Text(value='devrev_ref')))

Delta tables will be written under: devrev_ref


## 1. Pagination (unchanged) + landing pages in a Delta bronze table

Same cursor-loop as the plain-Python reference. The only addition: each page
is appended to a Delta table as it arrives, instead of only being accumulated
in memory — this is the standard "raw API dump to bronze" ingestion shape.

In [3]:
def fetch_all_pages_to_delta(fetch_page, table, start_cursor=None, max_pages=10_000):
    """Loop until there is no next cursor, appending each page straight to Delta.
    fetch_page(cursor) -> {"items": [...], "next_cursor": "abc" or None}
    Returns the total row count written."""
    cursor = start_cursor
    pages = 0
    total = 0
    while True:
        page = fetch_page(cursor)
        items = page["items"]
        if items:
            spark.createDataFrame(items).write.format("delta").mode("append").saveAsTable(table)
            total += len(items)
        cursor = page.get("next_cursor")
        pages += 1
        if not cursor:
            break
        if pages >= max_pages:
            raise RuntimeError("pagination exceeded max_pages (possible cursor loop)")
    return total

## 1b. Cursor watermark table — resumable pagination

A real ingestion job can fail mid-scan. Persisting the last successful cursor
in a tiny Delta control table means a rerun picks up where it left off instead
of re-fetching (and re-appending) everything from page 1.

In [4]:
WATERMARK_TABLE = qualified("ingest_watermark")

def init_watermark_table():
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {WATERMARK_TABLE} (
            source STRING, cursor STRING, updated_at TIMESTAMP
        ) USING DELTA
    """)

def read_watermark(source):
    init_watermark_table()
    row = spark.table(WATERMARK_TABLE).filter(F.col("source") == source).first()
    return row["cursor"] if row else None

def write_watermark(source, cursor):
    init_watermark_table()
    update_df = spark.createDataFrame([(source, cursor)], "source string, cursor string") \
        .withColumn("updated_at", F.current_timestamp())
    target = DeltaTable.forName(spark, WATERMARK_TABLE)
    (target.alias("t").merge(update_df.alias("s"), "t.source = s.source")
           .whenMatchedUpdateAll()
           .whenNotMatchedInsertAll()
           .execute())

def fetch_all_pages_resumable(fetch_page, table, source):
    """Same as fetch_all_pages_to_delta, but starts from the last checkpointed
    cursor and updates the checkpoint after every page (crash-safe resume)."""
    cursor = read_watermark(source)
    pages = 0
    total = 0
    while True:
        page = fetch_page(cursor)
        items = page["items"]
        if items:
            spark.createDataFrame(items).write.format("delta").mode("append").saveAsTable(table)
            total += len(items)
        cursor = page.get("next_cursor")
        write_watermark(source, cursor)          # checkpoint after every page, not just at the end
        pages += 1
        if not cursor:
            break
        if pages >= 10_000:
            raise RuntimeError("pagination exceeded max_pages (possible cursor loop)")
    return total

## 2. Retries & backoff (unchanged)

Retry-with-backoff and the `HTTPError`/`RETRYABLE_STATUS` pieces are pure
driver-side control flow around a single HTTP call — there's no Spark
equivalent, so this is identical to the plain-Python reference.

In [5]:
RETRYABLE_STATUS = {429, 500, 502, 503, 504}


class HTTPError(Exception):
    def __init__(self, status, message=""):
        super().__init__(f"HTTP {status} {message}")
        self.status = status


def retry_with_backoff(max_retries=5, base=0.5, cap=30.0, timeout_budget=60.0, sleep=time.sleep):
    """Retry transient HTTP errors with exponential backoff + full jitter."""
    def decorator(fn):
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            start = time.monotonic()
            attempt = 0
            while True:
                try:
                    return fn(*args, **kwargs)
                except HTTPError as e:
                    if e.status not in RETRYABLE_STATUS or attempt >= max_retries:
                        raise
                    ceiling = min(cap, base * (2 ** attempt))
                    delay = random.uniform(0, ceiling)   # FULL JITTER
                    if time.monotonic() - start + delay > timeout_budget:
                        raise
                    sleep(delay)
                    attempt += 1
        return wrapper
    return decorator

## 2b. Idempotent webhook processing -> Delta `MERGE INTO`

The Python version keeps an in-memory `{entity_id: highest_version}` dict.
That doesn't survive a job restart and doesn't scale past one process. The
Delta-native replacement: `MERGE INTO` the target table, only updating when
the incoming version is newer — exactly the same "never let a stale event
overwrite a newer one" rule, but durable and safe to apply from any number of
job runs.

In [6]:
EVENTS_TABLE = qualified("webhook_events")

def init_events_table():
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {EVENTS_TABLE} (
            id INT, version INT, payload STRING
        ) USING DELTA
    """)

def apply_events_idempotent(events):
    """events: list of {"id", "version", "payload"} dicts (one micro-batch).
    Duplicate or out-of-order (stale) events are silently no-ops, same as
    the Python IdempotentProcessor.process()."""
    init_events_table()
    source = spark.createDataFrame(events)
    target = DeltaTable.forName(spark, EVENTS_TABLE)
    (target.alias("t").merge(source.alias("s"), "t.id = s.id")
           .whenMatchedUpdate(condition="s.version > t.version",
                               set={"version": "s.version", "payload": "s.payload"})
           .whenNotMatchedInsertAll()
           .execute())

## 3. Rate limiting (unchanged)

`TokenBucket` stays as driver-side state, for the same reason pagination does:
a single sequential ingestion loop naturally has one rate to enforce. If you
ever *do* fan an API-calling workload out across executors (e.g. one call per
partition via `foreachPartition`), a per-driver `TokenBucket` no longer works —
you'd need a shared limiter (a small external service, or a Delta-table-backed
counter) that every executor checks against.

In [7]:
class TokenBucket:
    def __init__(self, rate, capacity):
        self.rate = rate
        self.capacity = capacity
        self.tokens = float(capacity)
        self.last = time.monotonic()
        self.lock = threading.Lock()

    def _refill(self):
        now = time.monotonic()
        elapsed = now - self.last
        self.tokens = min(self.capacity, self.tokens + elapsed * self.rate)
        self.last = now

    def acquire(self, n=1):
        while True:
            with self.lock:
                self._refill()
                if self.tokens >= n:
                    self.tokens -= n
                    return
                missing = n - self.tokens
                wait = missing / self.rate
            time.sleep(wait)

## Demo / self-test

### 1. Pagination -> Delta bronze table

In [8]:
DATA = [{"id": i, "value": i * 10} for i in range(1, 24)]   # 23 records to page through
BRONZE_TABLE = qualified("bronze_items")
spark.sql(f"DROP TABLE IF EXISTS {BRONZE_TABLE}")

def cursor_page(cursor):
    start = cursor or 0
    chunk = DATA[start:start + 10]
    nxt = start + 10 if start + 10 < len(DATA) else None
    return {"items": chunk, "next_cursor": nxt}

written = fetch_all_pages_to_delta(cursor_page, BRONZE_TABLE)
print("  wrote", written, "records to", BRONZE_TABLE)
assert spark.table(BRONZE_TABLE).count() == len(DATA)

  wrote 23 records to devrev_ref.bronze_items


### 1b. Resumable pagination via the watermark table

In [ ]:
RESUMABLE_TABLE = qualified("bronze_items_resumable")
spark.sql(f"DROP TABLE IF EXISTS {RESUMABLE_TABLE}")
spark.sql(f"DELETE FROM {WATERMARK_TABLE} WHERE source = 'demo'") if spark.catalog.tableExists(WATERMARK_TABLE) else None

# first run: simulate a crash after 2 pages by capping max_pages via a wrapping fetch_page
calls = {"n": 0}
def flaky_page(cursor):
    calls["n"] += 1
    if calls["n"] == 3:
        raise RuntimeError("simulated crash mid-scan")
    return cursor_page(cursor)

try:
    fetch_all_pages_resumable(flaky_page, RESUMABLE_TABLE, source="demo")
except RuntimeError as e:
    print("  simulated crash:", e)

after_crash = spark.table(RESUMABLE_TABLE).count()
print("  rows written before crash:", after_crash)
assert 0 < after_crash < len(DATA)

# rerun: resumes from the checkpointed cursor instead of starting over
fetch_all_pages_resumable(cursor_page, RESUMABLE_TABLE, source="demo")
final_count = spark.table(RESUMABLE_TABLE).count()
print("  rows written after resume:", final_count)
assert final_count == len(DATA)

  simulated crash: simulated crash mid-scan


  rows written before crash: 20


TypeError: can only concatenate str (not "int") to str

: 

### 2. Retry with backoff (force 429s, then succeed)

In [ ]:
calls2 = {"n": 0}
@retry_with_backoff(max_retries=5, base=0.01, cap=0.05, sleep=lambda d: None)
def flaky():
    calls2["n"] += 1
    if calls2["n"] < 3:
        raise HTTPError(429, "slow down")
    return "ok"
print("  result:", flaky(), "after", calls2["n"], "attempts")
assert calls2["n"] == 3

### 2b. Idempotent webhook processing via MERGE INTO

In [ ]:
spark.sql(f"DROP TABLE IF EXISTS {EVENTS_TABLE}")

apply_events_idempotent([{"id": 42, "version": 2, "payload": "v2 payload"}])
apply_events_idempotent([{"id": 42, "version": 1, "payload": "stale, should be ignored"}])
apply_events_idempotent([{"id": 42, "version": 2, "payload": "duplicate, should be a no-op"}])

row = spark.table(EVENTS_TABLE).filter("id = 42").first()
print("  final row:", row)
assert row["version"] == 2 and row["payload"] == "v2 payload"
assert spark.table(EVENTS_TABLE).count() == 1   # MERGE never created a duplicate row

### 3. Token bucket enforces the rate

In [ ]:
tb = TokenBucket(rate=20, capacity=5)
t0 = time.monotonic()
for _ in range(15):
    tb.acquire()
elapsed = time.monotonic() - t0
print(f"  15 requests (burst 5) took {elapsed:.2f}s (expected ~0.5s)")
assert 0.35 <= elapsed <= 0.9

print("\nALL CHECKS PASSED")